In [ ]:
# 1.1 Upload dataset
from google.colab import files
uploaded = files.upload()

Saving MSCA 691 Winter 2026 Exam - Classification data.xlsx to MSCA 691 Winter 2026 Exam - Classification data (2).xlsx


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_excel("MSCA 691 Winter 2026 Exam - Classification data.xlsx")

# **Explore**

**Define X and y**

In [ ]:
X = df.drop('Y', axis=1).values
y = df['Y'].values

#Conversion of labels

y = df['Y'].replace({1:0,2:1}).values

print("Unique y values:", np.unique(y))


Unique y values: [0 1]



**10-fold Stratified CV**

In [ ]:
from sklearn.model_selection import StratifiedKFold


In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# **Pipeline**

In [ ]:
from tensorflow.keras import backend as K
import gc

In [ ]:
def evaluate_keras_model(model_name, build_fn, X_data, y_data, sequence_model=False,
                         feature_selection=None, k_features=None,
                         epochs=20, batch_size=16):

    acc_list = []
    prec_list = []
    recall_list = []
    f1_list = []
    auc_list = []
    spec_list = []



    start_time = time.time()

    for fold_i, (train_rows, test_rows) in enumerate(skf.split(X_data, y_data), start=1):

        # 1) Split
        X_train, X_test = X_data[train_rows], X_data[test_rows]
        y_train, y_test = y_data[train_rows], y_data[test_rows]

        # 2) Feature selection

        if feature_selection == "mi":
            selector = SelectKBest(score_func=mutual_info_classif, k=k_features)
            X_train = selector.fit_transform(X_train, y_train)
            X_test = selector.transform(X_test)

        elif feature_selection == "sfs":
            selector = SequentialFeatureSelector(
                LogisticRegression(max_iter=1000),
                n_features_to_select=k_features,
                direction="forward",
                scoring="accuracy",
                cv=5
            )
            X_train = selector.fit_transform(X_train, y_train)
            X_test = selector.transform(X_test)

        # 2) Scale
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # 3) Reshape only for sequence models

        if sequence_model:
            X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
            X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))
            n_features = X_train.shape[2]
        else:
            n_features = X_train.shape[1]

        # 4) Build fresh model for this fold
        model = build_fn(n_features)

        # 5) Train
        model.fit(
            X_train, y_train,
            epochs=epochs,
            batch_size=batch_size,
            verbose=0
        )

        # 6) Predict
        y_prob = model.predict(X_test, verbose=0).ravel()
        y_pred = (y_prob >= 0.7).astype(int)

        # 7) Metrics
        acc_list.append(accuracy_score(y_test, y_pred))
        prec_list.append(precision_score(y_test, y_pred, zero_division=0))
        recall_list.append(recall_score(y_test, y_pred, zero_division=0))
        f1_list.append(f1_score(y_test, y_pred, zero_division=0))
        auc_list.append(roc_auc_score(y_test, y_prob))

        cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        print("Actual 0s:", np.sum(y_test == 0), "Actual 1s:", np.sum(y_test == 1))
        print("Pred  0s:", np.sum(y_pred == 0), "Pred  1s:", np.sum(y_pred == 1))
        print("Confusion matrix:")
        print(cm)
        print("TN =", tn, "FP =", fp, "FN =", fn, "TP =", tp)
        print("-" * 40)

        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        spec_list.append(specificity)


        #Delete model - My session keeps crashing because I'm using all RAM

        del model, X_train, X_test, y_train, y_test, y_prob, y_pred, cm
        K.clear_session()
        gc.collect()

    runtime = time.time() - start_time


    print(f"\n{model_name} SUMMARY:")
    print(f"     Runtime: {runtime:.2f}s")
    print(f"     accuracy   : {np.mean(acc_list):.4f}")
    print(f"     precision  : {np.mean(prec_list):.4f}")
    print(f"     recall     : {np.mean(recall_list):.4f}")
    print(f"     sensitivity: {np.mean(recall_list):.4f}")
    print(f"     specificity: {np.mean(spec_list):.4f}")
    print(f"     f1         : {np.mean(f1_list):.4f}")
    print(f"     roc_auc    : {np.mean(auc_list):.4f}")

    return {
        "Model": model_name,
        "Accuracy": np.mean(acc_list),
        "Sensitivity": np.mean(recall_list),
        "Specificity": np.mean(spec_list),
        "AUC": np.mean(auc_list),
        "F1": np.mean(f1_list)
    }

results = []


# **Evaluation**

In [ ]:
import time
from sklearn.feature_selection import SelectKBest, mutual_info_classif, SequentialFeatureSelector
from keras.models import Sequential
from keras.layers import SimpleRNN, LSTM, Bidirectional, GRU, Dense
from tensorflow.keras.utils import plot_model
from tensorflow.keras.layers import Dense, Dropout, Input
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import tensorflow as tf

In [ ]:
!pip install keras

##GRU

###GRU

In [ ]:
#Model
def build_GRU(n_features):
    model = Sequential()
    model.add(GRU(16, return_sequences=False, input_shape=(1, n_features )))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

#Evaluate

results.append(evaluate_keras_model(
    "GRU_SetA",
    build_GRU,
    X_data=X,
    y_data=y,
    feature_selection=None,
    sequence_model=True,
))
results.append(evaluate_keras_model(
    "GRU_SetB",
    build_GRU,
    X_data=X,
    y_data=y,
    feature_selection="mi",
    k_features=10,
    sequence_model=True
))
results.append(evaluate_keras_model(
    "GRU_SetC",
    build_GRU,
    X_data=X,
    y_data=y,
    feature_selection="sfs",
    k_features=10,
    sequence_model=True
))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Actual 0s: 5 Actual 1s: 13
Pred  0s: 9 Pred  1s: 9
Confusion matrix:
[[4 1]
 [5 8]]
TN = 4 FP = 1 FN = 5 TP = 8
----------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Actual 0s: 4 Actual 1s: 13
Pred  0s: 4 Pred  1s: 13
Confusion matrix:
[[ 2  2]
 [ 2 11]]
TN = 2 FP = 2 FN = 2 TP = 11
----------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

Actual 0s: 5 Actual 1s: 12
Pred  0s: 10 Pred  1s: 7
Confusion matrix:
[[4 1]
 [6 6]]
TN = 4 FP = 1 FN = 6 TP = 6
----------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Actual 0s: 5 Actual 1s: 12
Pred  0s: 9 Pred  1s: 8
Confusion matrix:
[[2 3]
 [7 5]]
TN = 2 FP = 3 FN = 7 TP = 5
----------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Actual 0s: 5 Actual 1s: 12
Pred  0s: 11 Pred  1s: 6
Confusion matrix:
[[2 3]
 [9 3]]
TN = 2 FP = 3 FN = 9 TP = 3
----------------------------------------

GRU_SetC SUMMARY:
     Runtime: 182.95s
     accuracy   : 0.5075
     precision  : 0.7541
     recall     : 0.4776
     sensitivity: 0.4776
     specificity: 0.5850
     f1         : 0.5815
     roc_auc    : 0.5444


###Stacked GRU

In [ ]:
#Model

def build_STACKEDGRU(n_features):
    model = Sequential()
    model.add(GRU(16, return_sequences=True, input_shape=(1, n_features)))
    model.add(GRU(8, return_sequences=False))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

#Evaluate

results.append(evaluate_keras_model(
    "STACKEDGRU_SetA",
    build_STACKEDGRU,
    X_data=X,
    y_data=y,
    feature_selection=None,
    sequence_model=True,
))
results.append(evaluate_keras_model(
    "STACKEDGRU_SetB",
    build_STACKEDGRU,
    X_data=X,
    y_data=y,
    feature_selection="mi",
    k_features=10,
    sequence_model=True
))
results.append(evaluate_keras_model(
    "STACKEDGRU_SetC",
    build_STACKEDGRU,
    X_data=X,
    y_data=y,
    feature_selection="sfs",
    k_features=10,
    sequence_model=True
))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Actual 0s: 5 Actual 1s: 13
Pred  0s: 6 Pred  1s: 12
Confusion matrix:
[[ 3  2]
 [ 3 10]]
TN = 3 FP = 2 FN = 3 TP = 10
----------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Actual 0s: 4 Actual 1s: 13
Pred  0s: 3 Pred  1s: 14
Confusion matrix:
[[ 1  3]
 [ 2 11]]
TN = 1 FP = 3 FN = 2 TP = 11
----------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

Actual 0s: 5 Actual 1s: 12
Pred  0s: 9 Pred  1s: 8
Confusion matrix:
[[4 1]
 [5 7]]
TN = 4 FP = 1 FN = 5 TP = 7
----------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Actual 0s: 5 Actual 1s: 12
Pred  0s: 7 Pred  1s: 10
Confusion matrix:
[[2 3]
 [5 7]]
TN = 2 FP = 3 FN = 5 TP = 7
----------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Actual 0s: 5 Actual 1s: 12
Pred  0s: 8 Pred  1s: 9
Confusion matrix:
[[3 2]
 [5 7]]
TN = 3 FP = 2 FN = 5 TP = 7
----------------------------------------

STACKEDGRU_SetC SUMMARY:
     Runtime: 218.06s
     accuracy   : 0.5725
     precision  : 0.7631
     recall     : 0.5974
     sensitivity: 0.5974
     specificity: 0.4950
     f1         : 0.6657
     roc_auc    : 0.5679


###BiGRU

In [ ]:
#Model

def build_biGRU(n_features):
    model = Sequential()
    model.add(Bidirectional(GRU(units=16, return_sequences=True), input_shape=(1, n_features)))
    model.add (Bidirectional(GRU(units=8)))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

#Evaluate

results.append(evaluate_keras_model(
    "biGRU_SetA",
    build_biGRU,
    X_data=X,
    y_data=y,
    feature_selection=None,
    sequence_model=True,
))
results.append(evaluate_keras_model(
    "biGRU_SetB",
    build_biGRU,
    X_data=X,
    y_data=y,
    feature_selection="mi",
    k_features=10,
    sequence_model=True
))
results.append(evaluate_keras_model(
    "biGRU_SetC",
    build_biGRU,
    X_data=X,
    y_data=y,
    feature_selection="sfs",
    k_features=10,
    sequence_model=True
))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Actual 0s: 5 Actual 1s: 13
Pred  0s: 5 Pred  1s: 13
Confusion matrix:
[[ 3  2]
 [ 2 11]]
TN = 3 FP = 2 FN = 2 TP = 11
----------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Actual 0s: 4 Actual 1s: 13
Pred  0s: 5 Pred  1s: 12
Confusion matrix:
[[ 2  2]
 [ 3 10]]
TN = 2 FP = 2 FN = 3 TP = 10
----------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

Actual 0s: 5 Actual 1s: 12
Pred  0s: 7 Pred  1s: 10
Confusion matrix:
[[3 2]
 [4 8]]
TN = 3 FP = 2 FN = 4 TP = 8
----------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Actual 0s: 5 Actual 1s: 12
Pred  0s: 6 Pred  1s: 11
Confusion matrix:
[[1 4]
 [5 7]]
TN = 1 FP = 4 FN = 5 TP = 7
----------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Actual 0s: 5 Actual 1s: 12
Pred  0s: 8 Pred  1s: 9
Confusion matrix:
[[1 4]
 [7 5]]
TN = 1 FP = 4 FN = 7 TP = 5
----------------------------------------

biGRU_SetC SUMMARY:
     Runtime: 263.38s
     accuracy   : 0.5784
     precision  : 0.7430
     recall     : 0.6365
     sensitivity: 0.6365
     specificity: 0.4150
     f1         : 0.6832
     roc_auc    : 0.5594


In [ ]:
results_df = pd.DataFrame(results)

#Results

results_df = results_df.round(4)


print("Original:")
results_df



Original:


,Model,Accuracy,Sensitivity,Specificity,AUC,F1
0,GRU_SetA,0.5373,0.5571,0.470,0.5517,0.6281
1,GRU_SetB,0.5314,0.4853,0.655,0.5623,0.5853
2,GRU_SetC,0.5075,0.4776,0.585,0.5444,0.5815
3,STACKEDGRU_SetA,0.6134,0.6788,0.430,0.5990,0.7142
4,STACKEDGRU_SetB,0.5961,0.6474,0.465,0.5719,0.6897
5,STACKEDGRU_SetC,0.5725,0.5974,0.495,0.5679,0.6657
6,biGRU_SetA,0.6366,0.6795,0.515,0.6179,0.7278
7,biGRU_SetB,0.5843,0.6545,0.390,0.5523,0.6901
8,biGRU_SetC,0.5784,0.6365,0.415,0.5594,0.6832


In [ ]:
#Results by AUC

results_dfAUC=results_df.copy()

results_dfAUC=results_dfAUC.sort_values(by="AUC", ascending=False)

print("\nSorted by AUC (descending):")
results_dfAUC


Sorted by AUC (descending):


,Model,Accuracy,Sensitivity,Specificity,AUC,F1
6,biGRU_SetA,0.6366,0.6795,0.515,0.6179,0.7278
3,STACKEDGRU_SetA,0.6134,0.6788,0.430,0.5990,0.7142
4,STACKEDGRU_SetB,0.5961,0.6474,0.465,0.5719,0.6897
5,STACKEDGRU_SetC,0.5725,0.5974,0.495,0.5679,0.6657
1,GRU_SetB,0.5314,0.4853,0.655,0.5623,0.5853
8,biGRU_SetC,0.5784,0.6365,0.415,0.5594,0.6832
7,biGRU_SetB,0.5843,0.6545,0.390,0.5523,0.6901
0,GRU_SetA,0.5373,0.5571,0.470,0.5517,0.6281
2,GRU_SetC,0.5075,0.4776,0.585,0.5444,0.5815


# **Hyperparameter Tuning**

**The top three models were selected based on AUC from the baseline comparison results:**

**Top 3 Models: StackedLSTM_SetA, DNN_SetA, biGRU_SetA:**

In [ ]:
import random
from tensorflow.keras.optimizers import Adam

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

**Building Models**

Blueprint for each model type

In [ ]:
#BiGRU
def build_biGRU_tuned(n_features, units1, units2, learning_rate):
    model = Sequential([
        Bidirectional(GRU(units1, return_sequences=True), input_shape=(1, n_features)),
        Bidirectional(GRU(units2)),
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

#Stacked GRU
def build_STACKEDGRU_tuned(n_features, units1, units2, learning_rate):
    model = Sequential([
        GRU(units1, return_sequences=True, input_shape=(1, n_features)),
        GRU(units2),
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

**Evaluation**

In [ ]:
def evaluate_tuned_model(model_name, build_fn, X_data, y_data,
                         sequence_model=False,
                         epochs=20, batch_size=16,
                         **build_kwargs):

    acc_list = []
    recall_list = []
    spec_list = []
    auc_list = []
    f1_list = []

    start_time = time.time()

    for fold_i, (train_rows, test_rows) in enumerate(skf.split(X_data, y_data), start=1):

        # Split
        X_train, X_test = X_data[train_rows], X_data[test_rows]
        y_train, y_test = y_data[train_rows], y_data[test_rows]

        # Scale inside each fold
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # Reshape only for sequence models
        if sequence_model:
            X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
            X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))
            n_features = X_train.shape[2]
        else:
            n_features = X_train.shape[1]

        # Build model using sampled hyperparameters
        model = build_fn(n_features, **build_kwargs)

        # Train
        model.fit(
            X_train, y_train,
            epochs=epochs,
            batch_size=batch_size,
            verbose=0
        )

        # Predict
        y_prob = model.predict(X_test, verbose=0).ravel()
        y_pred = (y_prob >= 0.7).astype(int)

        # Confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

        # Metrics
        acc_list.append(accuracy_score(y_test, y_pred))
        recall_list.append(recall_score(y_test, y_pred, zero_division=0))
        f1_list.append(f1_score(y_test, y_pred, zero_division=0))
        auc_list.append(roc_auc_score(y_test, y_prob))
        spec_list.append(tn / (tn + fp) if (tn + fp) > 0 else 0)

        # cleanup
        del model, X_train, X_test, y_train, y_test, y_prob, y_pred
        K.clear_session()
        gc.collect()

    runtime = time.time() - start_time

    return {
        "Model": model_name,
        "Accuracy": np.mean(acc_list),
        "Sensitivity": np.mean(recall_list),
        "Specificity": np.mean(spec_list),
        "AUC": np.mean(auc_list),
        "F1": np.mean(f1_list)
    }

**Random Search**

In [ ]:
def random_search_tuning(model_name, build_fn, X_data, y_data,
                         param_dist, n_iter=10, sequence_model=False):

    tuning_results = []

    for i in range(n_iter):
        # Randomly choose one value for each hyperparameter
        params = {k: random.choice(v) for k, v in param_dist.items()}

        # Evaluate that one sampled combination
        result = evaluate_tuned_model(
            model_name=f"{model_name}_tuned_{i+1}",
            build_fn=build_fn,
            X_data=X_data,
            y_data=y_data,
            sequence_model=sequence_model,
            epochs=params["epochs"],
            batch_size=params["batch_size"],
            **{k: v for k, v in params.items() if k not in ["epochs", "batch_size"]}
        )

        tuning_results.append(result)

        gc.collect()
        K.clear_session()

    # Convert all tuning results into a dataframe
    tuning_df = pd.DataFrame(tuning_results).round(4)

    # Sort by AUC from highest to lowest
    tuning_df = tuning_df.sort_values(by="AUC", ascending=False).reset_index(drop=True)

    return tuning_df

**Parameters**

In [ ]:
# biGRU hyperparameters
bigru_param_dist = {
    "units1": [8, 16, 32],
    "units2": [4, 8, 16],
    "learning_rate": [0.001, 0.0005],
    "epochs": [10, 20],
    "batch_size": [16, 32]
}

#STACKED GRU hyperparameters
stackedgru_param_dist = {
    "units1": [8, 16, 32],
    "units2": [4, 8, 16],
    "learning_rate": [0.001, 0.0005],
    "epochs": [10, 20],
    "batch_size": [16, 32]
}


**Run Tuning**

In [ ]:
#biGRU
bigru_tuning_df = random_search_tuning(
    model_name="biGRU_Set A",
    build_fn=build_biGRU_tuned,
    X_data=X,
    y_data=y,
    param_dist=bigru_param_dist,
    n_iter=10,
    sequence_model=True
)

print("biGRU tuning results:")
print(bigru_tuning_df)

# STACKED GRU

stackedgru_tuning_df = random_search_tuning(
    model_name="STACKEDGRU_SetA",
    build_fn=build_STACKEDGRU_tuned,
    X_data=X,
    y_data=y,
    param_dist=stackedgru_param_dist,
    n_iter=10,
    sequence_model=True
)

print("STACKED GRU tuning results:")
print(stackedgru_tuning_df)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `

# **Model Comparison**

In [ ]:
# Combine and Compare
combined_tuning_df = pd.concat(
    [bigru_tuning_df, stackedgru_tuning_df],
    ignore_index=True
)

# Sort by AUC (best at top)
combined_tuning_df = combined_tuning_df.sort_values(
    by="AUC", ascending=False
).reset_index(drop=True)

print("\nCombined tuning results (sorted by AUC):")
print(combined_tuning_df)

# Best Model
best_model = combined_tuning_df.iloc[0]

print("\nBest overall model:")
print(best_model)

# Clean comparison table
print("\nClean comparison table:")
print(combined_tuning_df[[
    "Model", "AUC", "Accuracy", "F1", "Sensitivity", "Specificity"
]])